# 🔗 Comprehensive LangChain Tutorial

This notebook provides a complete guide to using **LangChain**, a powerful open-source framework for building applications with Large Language Models (LLMs). It covers core components, advanced features, and practical examples for e-commerce and other applications. The notebook preserves the original code while adding theoretical explanations, comments, and additional functionalities.

## What is LangChain?
LangChain is a framework that simplifies the development of context-aware, reasoning-based applications using LLMs. It provides tools for:
- **Prompt Management**: Crafting and optimizing prompts for LLMs.
- **Memory**: Maintaining conversational context across interactions.
- **Retrieval-Augmented Generation (RAG)**: Combining LLMs with external data sources.
- **Agents**: Enabling LLMs to make decisions and interact with tools.
- **Chains**: Composing modular workflows for complex tasks.
- **Tools**: Integrating external APIs and services.

## Objectives
- Demonstrate all major LangChain functionalities.
- Provide theoretical context for each component.
- Include practical examples with comments.
- Preserve and enhance the original notebook's code.

## Prerequisites
- Python 3.10+
- Install dependencies: `pip install langchain langchain-community langchain-huggingface langchain-ollama chromadb sentence-transformers`
- Sample text file (`sample.txt`) for RAG examples.

## Structure
1. **Setup and Basic Components**
2. **Retrieval-Augmented Generation (RAG)**
3. **Conversational Memory**
4. **Chains and Pipelines**
5. **Agents and Tools**
6. **Advanced Features**
7. **Tracing and Monitoring**
8. **Batch Processing and Parallel Execution**
9. **Custom Components**

Let's dive in!

## 1. Setup and Basic Components

**Theory**: LangChain requires initializing LLMs, embeddings, and document loaders to process and interact with data. The original notebook uses HuggingFace embeddings and Ollama's LLaMA3 model for lightweight, open-source processing. This section sets up the environment and demonstrates basic components.

In [ ]:
# Uninstall conflicting packages (as in original notebook)
!pip uninstall -y langchain-openai chromadb

# Install required packages
!pip install langchain langchain-community langchain-huggingface langchain-ollama chromadb sentence-transformers

In [ ]:
# Import core LangChain components
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM

# Initialize document loader and splitter
# Theory: TextLoader reads text files, and RecursiveCharacterTextSplitter chunks documents for efficient processing.
loader = TextLoader('sample.txt')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Initialize embeddings
# Theory: Embeddings convert text to vectors for semantic search.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Initialize LLM
# Theory: Ollama provides local, open-source LLMs for cost-effective processing.
llm = OllamaLLM(model="llama3")

## 2. Retrieval-Augmented Generation (RAG)

**Theory**: RAG combines LLMs with external knowledge bases to provide contextually relevant answers. It involves:
- **Vector Stores**: Storing document embeddings (e.g., Chroma).
- **Retrievers**: Fetching relevant documents (e.g., vector-based or BM25).
- **QA Chains**: Combining retrieved documents with LLM responses.

The original notebook demonstrates basic RAG, ensemble retrievers, and contextual compression. We'll expand with multi-query retrieval and parent document retrieval.

In [ ]:
# Basic RAG Setup (Original Code)
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA

# Create vector store and retriever
vector_store = Chroma.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Run query (Original Code)
query = "who are you"
result = qa_chain({"query": query})
print("Answer:", result['result'])
print("Sources:", [doc.metadata for doc in result["source_documents"]])

In [ ]:
# Ensemble Retriever (Original Code)
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# Create BM25 retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2

# Combine vector and BM25 retrievers
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

# Create RAG chain with ensemble retriever
ensemble_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever,
    return_source_documents=True
)

# Run query
query = "What workflows does LangChain support?"
result = ensemble_qa_chain.invoke({"query": query})
print("Answer:", result["result"])
print("Sources:", [doc.page_content[:100] for doc in result["source_documents"]])

In [ ]:
# Contextual Compression (Original Code)
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Create compressor
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

# Create RAG chain with compression
compressed_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True
)

# Run query
query = "What does LangChain support for retrieval?"
result = compressed_qa_chain.invoke({"query": query})
print("Answer:", result["result"])
print("Sources:", [doc.page_content for doc in result["source_documents"]])

In [ ]:
# Multi-Query Retriever (New Addition)
from langchain.retrievers.multi_query import MultiQueryRetriever

# Create multi-query retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

# Create RAG chain with multi-query retriever
multi_query_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=multi_query_retriever,
    return_source_documents=True
)

# Run query
query = "What are the main features of LangChain?"
result = multi_query_qa_chain.invoke({"query": query})
print("Answer:", result["result"])
print("Sources:", [doc.page_content[:100] for doc in result["source_documents"]])

In [ ]:
# Parent Document Retriever (New Addition)
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

# Create parent splitter for larger chunks
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# Initialize parent document retriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=InMemoryStore(),
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

# Add documents
parent_retriever.add_documents(documents)

# Create RAG chain with parent document retriever
parent_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=parent_retriever,
    return_source_documents=True
)

# Run query
query = "What is LangChain's approach to memory?"
result = parent_qa_chain.invoke({"query": query})
print("Answer:", result["result"])
print("Sources:", [doc.page_content[:100] for doc in result["source_documents"]])

## 3. Conversational Memory

**Theory**: LangChain's memory components maintain conversational context, enabling coherent interactions. Types include:
- **ConversationBufferMemory**: Stores all messages.
- **ConversationSummaryMemory**: Summarizes conversations.
- **ConversationBufferWindowMemory**: Stores a fixed number of recent messages.

The original notebook includes a conversational chain with buffer memory. We'll add summary and window memory examples.

In [ ]:
# Conversational Retrieval Chain with Buffer Memory (Original Code)
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# Initialize memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Create conversational chain
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True
)

# Run query
result = conversational_chain({"question": "Can you explain the main topic further?"})
print("Answer:", result["answer"])

In [ ]:
# Conversation Summary Memory (New Addition)
from langchain.memory import ConversationSummaryMemory

# Initialize summary memory
summary_memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history")

# Create conversational chain with summary memory
summary_conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=summary_memory,
    return_source_documents=True
)

# Run queries to demonstrate memory
result1 = summary_conversational_chain({"question": "What is LangChain?"})
result2 = summary_conversational_chain({"question": "Can you elaborate on its features?"})
print("Answer 1:", result1["answer"])
print("Answer 2:", result2["answer"])

In [ ]:
# Conversation Buffer Window Memory (New Addition)
from langchain.memory import ConversationBufferWindowMemory

# Initialize window memory (stores last 2 interactions)
window_memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history", return_messages=True)

# Create conversational chain with window memory
window_conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=window_memory,
    return_source_documents=True
)

# Run queries
result1 = window_conversational_chain({"question": "What does LangChain do?"})
result2 = window_conversational_chain({"question": "What about its agents?"})
print("Answer 1:", result1["answer"])
print("Answer 2:", result2["answer"])

## 4. Chains and Pipelines

**Theory**: Chains combine LLMs, prompts, and other components into workflows. Types include:
- **LLMChain**: Simple prompt-LLM interaction.
- **SequentialChain**: Sequential execution of chains.
- **RunnableParallel**: Parallel execution of tasks.
- **RunnableBranch**: Conditional routing.

The original notebook includes SequentialChain, RunnableParallel, and RunnableBranch examples. We'll add custom chains and LCEL (LangChain Expression Language) examples.

In [ ]:
# Sequential Chain (Original Code)
from langchain.chains import SequentialChain, LLMChain
from langchain.prompts import PromptTemplate

# First chain: Summarize
summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize the following in 50 words: {text}"
)
summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="summary")

# Second chain: Extract keywords
keywords_prompt = PromptTemplate(
    input_variables=["summary"],
    template="Extract 3 keywords from this summary: {summary}"
)
keywords_chain = LLMChain(llm=llm, prompt=keywords_prompt, output_key="keywords")

# Combine chains
overall_chain = SequentialChain(
    chains=[summary_chain, keywords_chain],
    input_variables=["text"],
    output_variables=["summary", "keywords"]
)

# Run the chain
text = "LangChain is a framework for building applications with large language models. It supports retrieval-augmented generation, agents, and memory for context-aware, reasoning-based apps."
result = overall_chain({"text": text})
print("Summary:", result["summary"])
print("Keywords:", result["keywords"])

In [ ]:
# Parallel Chain (Original Code)
from langchain_core.runnables import RunnableParallel

# Create parallel chain
chain = RunnableParallel(
    summary=summary_prompt | llm,
    keywords=keywords_prompt | llm
)

# Run chain
text = "LangChain supports metadata filtering and compression for efficient retrieval."
result = chain.invoke({"text": text})
print("Summary:", result["summary"])
print("Keywords:", result["keywords"])

In [ ]:
# Runnable Branch (Original Code)
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

# Direct LLM chain
direct_prompt = PromptTemplate(
    input_variables=["query"],
    template="Answer this general question: {query}"
)
direct_chain = direct_prompt | llm

# Classifier to determine query type
classifier_prompt = PromptTemplate(
    input_variables=["query"],
    template="Is this query about LangChain's features? Answer 'Yes' or 'No': {query}"
)
classifier_chain = classifier_prompt | llm | (lambda x: x.strip() == "Yes")

# Create branch
branch = RunnableBranch(
    (classifier_chain, qa_chain),
    direct_chain
)

# Run queries
queries = [
    "What workflows does LangChain support?",
    "What is the capital of France?"
]
for query in queries:
    result = branch.invoke({"query": query})
    print(f"Query: {query}\nAnswer: {result}\n")

In [ ]:
# Custom Chain (New Addition)
from langchain.chains.base import Chain
from typing import Dict, Any

class CustomSentimentChain(Chain):
    def __init__(self, llm):
        self.llm = llm
        self.sentiment_prompt = PromptTemplate(
            input_variables=["text"],
            template="Classify the sentiment of this text as Positive, Negative, or Neutral: {text}"
        )
        self.response_prompt = PromptTemplate(
            input_variables=["text", "sentiment"],
            template="Given the text '{text}' with {sentiment} sentiment, provide a suitable response."
        )

    @property
    def input_keys(self):
        return ["text"]

    @property
    def output_keys(self):
        return ["response"]

    def _call(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        text = inputs["text"]
        sentiment = self.llm(self.sentiment_prompt.format(text=text)).strip().split(":")[-1].strip()
        response = self.llm(self.response_prompt.format(text=text, sentiment=sentiment))
        return {"response": response}

# Create and run custom chain
custom_chain = CustomSentimentChain(llm=llm)
text = "I'm really excited about LangChain!"
result = custom_chain.invoke({"text": text})
print("Response:", result["response"])

## 5. Agents and Tools

**Theory**: Agents enable LLMs to make decisions and interact with external tools (e.g., search engines, APIs). LangChain supports:
- **React Agents**: Reason and act based on observations.
- **Toolkits**: Pre-built tool collections (e.g., Wikipedia, SerpAPI).
- **Custom Tools**: User-defined functions.

The original notebook doesn't include agents. We'll add examples using React agents and custom tools.

In [ ]:
# React Agent with Tools (New Addition)
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.tools import DuckDuckGoSearchRun

# Initialize search tool
search = DuckDuckGoSearchRun()

# Define tools
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="Useful for answering questions about current events or general knowledge"
    )
]

# Initialize agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
query = "What are the latest developments in LangChain?"
result = agent.run(query)
print("Answer:", result)

In [ ]:
# Custom Tool (New Addition)
from langchain.tools import tool

@tool
def calculate_discount(price: float, discount_percentage: float) -> str:
    """Calculate the discounted price given an original price and discount percentage."""
    discount = price * (discount_percentage / 100)
    final_price = price - discount
    return f"Original price: ${price:.2f}, Discount: ${discount:.2f}, Final price: ${final_price:.2f}"

# Initialize agent with custom tool
tools = [calculate_discount]
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
query = "Calculate the price of a $100 item with a 20% discount"
result = agent.run(query)
print("Answer:", result)

## 6. Advanced Features

**Theory**: LangChain offers advanced features like few-shot learning, custom prompts, and output parsing. The original notebook includes few-shot prompting. We'll add structured output parsing and custom prompt engineering.

In [ ]:
# Few-Shot Prompting (Original Code)
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from langchain.prompts.example_selector import LengthBasedExampleSelector

# Define examples
examples = [
    {"query": "How does RAG work in LangChain?", "label": "Technical"},
    {"query": "What is LangChain?", "label": "General"},
    {"query": "Explain embeddings.", "label": "Technical"},
    {"query": "Who created LangChain?", "label": "General"}
]

# Example prompt
example_prompt = PromptTemplate(
    input_variables=["query", "label"],
    template="Query: {query}\nLabel: {label}\n"
)

# Example selector
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=100
)

# Main prompt
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Classify the following query as Technical or General:",
    suffix="Query: {query}\nLabel:",
    input_variables=["query"]
)

# Create chain
chain = prompt | llm

# Run query
query = "How do agents work in LangChain?"
result = chain.invoke({"query": query})
print("Result:", result.strip())

In [ ]:
# Structured Output Parsing (New Addition)
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.prompts import PromptTemplate

# Define response schemas
response_schemas = [
    ResponseSchema(name="answer", description="The answer to the query"),
    ResponseSchema(name="confidence", description="Confidence score (0-1)")
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# Create prompt with format instructions
prompt = PromptTemplate(
    input_variables=["query"],
    template="Answer the query and provide a confidence score:\n{query}\n{format_instructions}",
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

# Create chain
chain = prompt | llm | output_parser

# Run query
query = "What is LangChain?"
result = chain.invoke({"query": query})
print("Structured Output:", result)

## 7. Tracing and Monitoring

**Theory**: LangChain integrates with LangSmith for tracing and debugging chains. The original notebook attempts LangSmith integration but encounters an error. We'll provide a corrected version and add custom callbacks.

In [ ]:
# Custom Callback (Original Code)
from langchain.callbacks.base import BaseCallbackHandler
import time

class LoggingCallback(BaseCallbackHandler):
    def on_chain_start(self, serialized, inputs, **kwargs):
        self.start_time = time.time()
        print(f"Chain started with inputs: {inputs}")

    def on_chain_end(self, outputs, **kwargs):
        elapsed = time.time() - self.start_time
        print(f"Chain completed in {elapsed:.2f}s. Output: {outputs}")

# Create chain with callbacks
callback = LoggingCallback()
chain = prompt | llm

# Run chain
text = "LangChain is a framework for building applications with LLMs."
result = chain.invoke({"text": text}, config={"callbacks": [callback]})
print("Summary:", result)

In [ ]:
# LangSmith Tracing (Corrected)
from langsmith import Client
from langchain.callbacks.manager import trace

# Initialize LangSmith client (requires API key)
# Set environment variables: export LANGCHAIN_API_KEY=your_api_key
client = Client()

# Create traceable function
@trace(name="qa_chain")
def run_query(query):
    return qa_chain.invoke({"query": query})

# Run query with tracing
query = "What is LangChain?"
result = run_query(query)
print("Answer:", result["result"])

## 8. Batch Processing and Parallel Execution

**Theory**: LangChain supports batch processing for handling multiple inputs efficiently. The original notebook includes batch processing for RAG queries. We'll add asynchronous processing.

In [ ]:
# Batch Processing (Original Code)
queries = [
    {"query": "What workflows does LangChain support?"},
    {"query": "What is LangChain?"}
]
results = qa_chain.batch(queries)
for query, result in zip(queries, results):
    print(f"Query: {query['query']}\nAnswer: {result['result']}\n")

In [ ]:
# Asynchronous Processing (New Addition)
import asyncio

async def run_async_queries(queries):
    tasks = [qa_chain.ainvoke(query) for query in queries]
    results = await asyncio.gather(*tasks)
    return results

# Run async queries
queries = [
    {"query": "What is LangChain?"},
    {"query": "What are its main features?"}
]
results = asyncio.run(run_async_queries(queries))
for query, result in zip(queries, results):
    print(f"Query: {query['query']}\nAnswer: {result['result']}\n")

## 9. Custom Components

**Theory**: LangChain allows creating custom components like retrievers, tools, and chains. The original notebook includes a custom sentiment chain. We'll add a custom retriever.

In [ ]:
# Custom Retriever (New Addition)
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document

class CustomKeywordRetriever(BaseRetriever):
    def __init__(self, documents, keywords):
        self.documents = documents
        self.keywords = keywords

    def _get_relevant_documents(self, query, *, run_manager=None):
        relevant_docs = []
        for doc in self.documents:
            if any(keyword.lower() in doc.page_content.lower() for keyword in self.keywords):
                relevant_docs.append(doc)
        return relevant_docs[:2]

# Create custom retriever
custom_retriever = CustomKeywordRetriever(documents=chunks, keywords=["LangChain", "RAG"])

# Create RAG chain with custom retriever
custom_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=custom_retriever,
    return_source_documents=True
)

# Run query
query = "What is LangChain?"
result = custom_qa_chain.invoke({"query": query})
print("Answer:", result["result"])

## Conclusion

This notebook covers all major LangChain functionalities, including RAG, memory, chains, agents, tools, advanced features, tracing, batch processing, and custom components. It preserves the original code while adding theoretical explanations and new examples. Use this as a reference for building sophisticated LLM applications.

For further exploration, check the [LangChain Documentation](https://python.langchain.com/docs/) and experiment with different LLMs, embeddings, and tools.